In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
# little function to define the file root on different machines
def find_f_root(start_path: Path = Path.cwd(), anchor: str = "CASA0004_work") -> Path:
    """
    Traverse up from the start_path until the anchor folder is found. Returns the path to the anchor folder.
    """
    for parent in [start_path] + list(start_path.parents):
        if parent.name == anchor:
            return parent
    raise FileNotFoundError(f"Anchor folder '{anchor}' not found in path hierarchy.")
  
f_root = find_f_root()

In [ ]:
fine_geogr_source = gpd.read_file(f_root / "data/geographies/ltla/Local_Authority_Districts_December_2023_Boundaries_UK_BFC_-643602861855028479/LAD_DEC_2023_UK_BFC.shp")
fine_geogr_source = fine_geogr_source.set_crs(epsg=27700)

In [ ]:
fine_geogr_source

In [ ]:
fine_geogr_source = fine_geogr_source[
    fine_geogr_source["LAD23CD"]
    .astype("string")
    .str.startswith(("E", "W"), na=False)
].copy()

fine_geogr_source.drop(
    columns=["LAD23NMW", "BNG_E","BNG_N","LONG", "LAT","GlobalID"],
    inplace=True
)

In [ ]:
fine_geogr_source

In [ ]:
lad_region_lookup = pd.read_csv(
    f_root
    / "data/geographies/ltla/Local_Authority_District_to_Region_(December_2023)_Lookup_in_England.csv",
    usecols=["LAD23CD", "RGN23NM"]
)

In [ ]:
lad_region_lookup

In [ ]:
fine_geogr_source = (
    fine_geogr_source
    .merge(
        lad_region_lookup,
        left_on="LAD23CD",
        right_on="LAD23CD",
        how="left",          # English LADs match; Welsh LADs come back NA
        validate="many_to_one"
    )
)

# English LADs now have their region; give Welsh LADs "Wales"
fine_geogr_source["RGN23NM"] = fine_geogr_source["RGN23NM"].fillna("Wales")

# Rename the region column to the short name you use elsewhere
fine_geogr_source = fine_geogr_source.rename(columns={"RGN23NM": "gor"})

In [ ]:
cols_to_drop = [
    "LAD25NMW",
    "RGN23CD",
    "FID",
    "LONG",
    "LAT",
    "GlobalID"
]

fine_geogr_source.drop(
    columns=[col for col in cols_to_drop if col in fine_geogr_source.columns],
    inplace=True,
)

In [ ]:
fine_geogr_source

In [ ]:
fine_geogr_source.plot()

In [ ]:
fine_geogr_source.gor.value_counts()

In [ ]:
fine_geogr_source.info()

In [ ]:
fine_geogr_source.to_file(
    f_root / "data/geographies/ltla/export_small_geogr_ew.shp",
    driver="ESRI Shapefile"
)